# 00. ELT Pipeline — 원본 적재

원본 Excel 90개를 MySQL의 `raw_*` 테이블에 적재한다. `raw_*`에는 분석에 필요한 원본 업무 컬럼만 저장하고, 파일·배치 적재 이력은 `meta_load_*` 테이블로 분리한다.

이 노트북은 원본 파일을 수정하지 않는다. 새 스키마로 다시 적재해야 할 때만 `sql/00_rebuild_raw_schema.sql`을 먼저 실행한다.

In [1]:
from pathlib import Path
import hashlib
import os
import re

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

load_dotenv(ROOT / '.env')
DATA_DIR = Path(os.getenv('DATA_DIR', '')).expanduser()
SOURCE_ROOT = DATA_DIR / 'airportal'
SQL_FILE = ROOT / 'sql' / '00_elt_pipeline.sql'

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f'원본 폴더가 없음: {SOURCE_ROOT}')
if not SQL_FILE.exists():
    raise FileNotFoundError(f'테이블 생성 SQL이 없음: {SQL_FILE}')

db_config = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', '3306')),
    'database': os.getenv('DB_NAME'),
    'username': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
}
missing = [key for key in ['host', 'database', 'username', 'password'] if not db_config[key]]
if missing:
    raise ValueError(f'.env의 DB 설정이 비어 있음: {missing}')

In [2]:
DATASETS = [
    {'target': 'raw_route_passenger', 'pattern': 'transport_statistics/route/route_passenger_*.xlsx',
     'columns': ['노선1', '노선2', '년월', '여객(명)'],
     'rename': {'노선1': 'origin_airport_raw', '노선2': 'destination_airport_raw', '년월': 'period_raw', '여객(명)': 'passengers'}},
    {'target': 'raw_route_operations', 'pattern': 'transport_statistics/route/route_operations_*.xlsx',
     'columns': ['노선1', '노선2', '년월', '운항(편)'],
     'rename': {'노선1': 'origin_airport_raw', '노선2': 'destination_airport_raw', '년월': 'period_raw', '운항(편)': 'operations'}},
    {'target': 'raw_airline_passenger', 'pattern': 'transport_statistics/airline/airline_passenger_*.xlsx',
     'columns': ['항공사명', '년월', '여객(명)'],
     'rename': {'항공사명': 'airline_name_raw', '년월': 'period_raw', '여객(명)': 'passengers'}},
    {'target': 'raw_airline_operations', 'pattern': 'transport_statistics/airline/airline_operations_*.xlsx',
     'columns': ['항공사명', '년월', '운항(편)'],
     'rename': {'항공사명': 'airline_name_raw', '년월': 'period_raw', '운항(편)': 'operations'}},
    {'target': 'raw_airport_passenger', 'pattern': 'transport_statistics/airport/airport_passenger_*.xlsx',
     'columns': ['공항명', '년월', '여객(명)'],
     'rename': {'공항명': 'airport_name_raw', '년월': 'period_raw', '여객(명)': 'passengers'}},
    {'target': 'raw_country_passenger', 'pattern': 'transport_statistics/country/country_passenger_*.xlsx',
     'columns': ['지역', '국가', '년월', '여객(명)'],
     'rename': {'지역': 'region_raw', '국가': 'country_raw', '년월': 'period_raw', '여객(명)': 'passengers'}},
]

FLIGHT_DATASET = {
    'target': 'raw_departure_flight',
    'pattern': 'aircraft_departures/*/*.xlsx',
    'columns': ['출발/도착', '공항명', '항공사', '편명', '도착지', '일자', '계획시간', '예상시간', '출발시간', '구분', '상태', '지연원인'],
    'rename': {
        '출발/도착': 'direction_raw', '공항명': 'airport_name_raw', '항공사': 'airline_name_raw',
        '편명': 'flight_number_raw', '도착지': 'destination_raw', '일자': 'flight_date_raw',
        '계획시간': 'scheduled_time_raw', '예상시간': 'estimated_time_raw',
        '출발시간': 'actual_departure_time_raw', '구분': 'service_type_raw',
        '상태': 'status_raw', '지연원인': 'delay_reason_raw',
    },
}

for dataset in DATASETS + [FLIGHT_DATASET]:
    dataset['files'] = sorted(SOURCE_ROOT.glob(dataset['pattern']))

manifest = pd.DataFrame([
    {'target_table': dataset['target'], 'file_count': len(dataset['files'])}
    for dataset in DATASETS + [FLIGHT_DATASET]
])
display(manifest)
assert manifest['file_count'].sum() == 90, '예상한 원본 파일 수(90)와 다름'

,target_table,file_count
0,raw_route_passenger,3
1,raw_route_operations,3
2,raw_airline_passenger,3
3,raw_airline_operations,3
4,raw_airport_passenger,3
5,raw_country_passenger,3
6,raw_departure_flight,72


In [3]:
server_url = URL.create('mysql+pymysql', host=db_config['host'], port=db_config['port'],
                        username=db_config['username'], password=db_config['password'])
server_engine = create_engine(server_url, future=True)
with server_engine.begin() as connection:
    connection.execute(text(
        f"CREATE DATABASE IF NOT EXISTS `{db_config['database']}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
    ))

engine = create_engine(URL.create('mysql+pymysql', **db_config), pool_pre_ping=True, future=True)

# 스키마 SQL은 단순 DDL만 담으므로 주석을 제거한 뒤 세미콜론 기준으로 실행한다.
sql_text = '\n'.join(line for line in SQL_FILE.read_text(encoding='utf-8').splitlines() if not line.strip().startswith('--'))
for statement in sql_text.split(';'):
    if statement.strip():
        with engine.begin() as connection:
            connection.execute(text(statement))

print(f"Ready: {db_config['database']}")

Ready: korea_outbound_travel


In [4]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def source_period(path: Path, is_flight_log: bool):
    # 파일명에서 적재 이력용 기간만 추출한다. raw 테이블에는 넣지 않는다.
    match = re.search(r'(20\d{2})(?:[_-](0?[1-9]|1[0-2]))?', path.stem)
    if not match:
        raise ValueError(f'파일명에서 연월을 찾을 수 없음: {path.name}')
    return int(match.group(1)), int(match.group(2)) if is_flight_log and match.group(2) else None

def clean_text(value):
    return None if pd.isna(value) else str(value).strip()

def prepare_frame(path: Path, dataset: dict) -> pd.DataFrame:
    source = pd.read_excel(path, dtype=object)
    missing_columns = set(dataset['columns']) - set(source.columns)
    if missing_columns:
        raise ValueError(f'{path.name}의 필수 컬럼 누락: {sorted(missing_columns)}')

    # raw에는 업무 원본 컬럼만 남긴다. 파일 경로·행 번호는 meta_load_file에서만 관리한다.
    frame = source.loc[:, dataset['columns']].rename(columns=dataset['rename']).copy()
    for column in frame.columns:
        if column not in {'passengers', 'operations'}:
            frame[column] = frame[column].map(clean_text)
    for column in {'passengers', 'operations'} & set(frame.columns):
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    return frame

def existing_file_record(connection, source_file: str, target: str):
    return connection.execute(text(
        'SELECT source_sha256, status FROM meta_load_file WHERE source_file = :source_file AND target_table = :target_table'
    ), {'source_file': source_file, 'target_table': target}).mappings().first()

def record_loaded_file(connection, source_file, target, source_hash, year, month, source_row_count, batch_id):
    connection.execute(text(
        'INSERT INTO meta_load_file '
        '(source_file, target_table, source_sha256, source_year, source_month, source_rows, loaded_rows, batch_id, status) '
        'VALUES (:source_file, :target_table, :source_sha256, :source_year, :source_month, :source_rows, :loaded_rows, :batch_id, :status)'
    ), {
        'source_file': source_file, 'target_table': target, 'source_sha256': source_hash,
        'source_year': year, 'source_month': month, 'source_rows': source_row_count,
        'loaded_rows': source_row_count, 'batch_id': batch_id, 'status': 'loaded',
    })

def load_one(dataset: dict, path: Path, batch_id: int):
    target = dataset['target']
    source_file = path.relative_to(ROOT).as_posix()
    source_hash = file_sha256(path)

    with engine.connect() as connection:
        existing = existing_file_record(connection, source_file, target)
    if existing and existing['status'] == 'loaded' and existing['source_sha256'] == source_hash:
        return {'target_table': target, 'result': 'skipped', 'rows': 0}
    if existing and existing['status'] == 'loaded':
        raise RuntimeError(f'이미 적재한 원본 파일 내용이 변경됨: {source_file}')

    frame = prepare_frame(path, dataset)
    source_row_count = len(frame)
    year, month = source_period(path, target == 'raw_departure_flight')

    # 파일 데이터와 적재 이력을 하나의 트랜잭션으로 저장해 부분 적재를 남기지 않는다.
    with engine.begin() as connection:
        frame.to_sql(target, connection, if_exists='append', index=False, chunksize=5_000, method='multi')
        record_loaded_file(connection, source_file, target, source_hash, year, month, source_row_count, batch_id)

    return {'target_table': target, 'result': 'loaded', 'rows': source_row_count}

In [5]:
with engine.begin() as connection:
    batch_id = connection.execute(text(
        "INSERT INTO meta_load_batch (pipeline_name, source_root, status) VALUES ('00_elt_pipeline', :source_root, 'running')"
    ), {'source_root': SOURCE_ROOT.as_posix()}).lastrowid

results = []
try:
    for dataset in DATASETS + [FLIGHT_DATASET]:
        for path in dataset['files']:
            results.append(load_one(dataset, path, batch_id))
except Exception as error:
    with engine.begin() as connection:
        connection.execute(text(
            "UPDATE meta_load_batch SET status = 'failed', completed_at = CURRENT_TIMESTAMP, note = :note WHERE batch_id = :batch_id"
        ), {'note': str(error)[:1000], 'batch_id': batch_id})
    raise
else:
    with engine.begin() as connection:
        connection.execute(text(
            "UPDATE meta_load_batch SET status = 'loaded', completed_at = CURRENT_TIMESTAMP WHERE batch_id = :batch_id"
        ), {'batch_id': batch_id})

display(pd.DataFrame(results).groupby(['target_table', 'result'], as_index=False)['rows'].sum())

,target_table,result,rows
0,raw_airline_operations,skipped,0
1,raw_airline_passenger,skipped,0
2,raw_airport_passenger,skipped,0
3,raw_country_passenger,skipped,0
4,raw_departure_flight,skipped,0
5,raw_route_operations,skipped,0
6,raw_route_passenger,skipped,0


In [6]:
# 분석용 raw 테이블에는 원본 업무 컬럼만 존재하는지 행 수와 함께 확인한다.
raw_counts = pd.read_sql(text('''
SELECT 'raw_route_passenger' AS table_name, COUNT(*) AS row_count FROM raw_route_passenger
UNION ALL SELECT 'raw_route_operations', COUNT(*) FROM raw_route_operations
UNION ALL SELECT 'raw_airline_passenger', COUNT(*) FROM raw_airline_passenger
UNION ALL SELECT 'raw_airline_operations', COUNT(*) FROM raw_airline_operations
UNION ALL SELECT 'raw_airport_passenger', COUNT(*) FROM raw_airport_passenger
UNION ALL SELECT 'raw_country_passenger', COUNT(*) FROM raw_country_passenger
UNION ALL SELECT 'raw_departure_flight', COUNT(*) FROM raw_departure_flight
'''), engine)
display(raw_counts)

,table_name,row_count
0,raw_route_passenger,9670
1,raw_route_operations,9670
2,raw_airline_passenger,3951
3,raw_airline_operations,3951
4,raw_airport_passenger,301
5,raw_country_passenger,2083
6,raw_departure_flight,793865
